# 06 — Deep Learning Preprocessing
## CATIVE Phase 2: Hybrid Neural Network

### Architecture Overview
Our DL model uses two input streams that are fused before the final classifier:

```
top_hiring_roles (text)
        ↓
  DistilBERT tokeniser
        ↓
  [CLS] embedding (768-d)
        ↓
  Dense(128) + GELU
        ↓
   Dropout(0.3)          Tabular features (numeric + encoded)
        ↓                        ↓
   Text vector (128)    MLP [256→128→64] + BatchNorm + Dropout(0.3)
        ↓                        ↓
         ←──── Concatenate (128 + 64 = 192) ────→
                        ↓
                  Dense(64) + GELU
                        ↓
                  Softmax(3)
```

**Why DistilBERT?** DistilBERT (Sanh et al., 2019) retains 97% of BERT's performance at 40% less parameters. For our short role strings (avg ~5 tokens), full BERT is overkill. The [CLS] token embedding captures the semantic meaning of the role combination.

**Why a fusion architecture?** The roles text and structured features carry *complementary* information — the text captures *what talent is sought*, the structured features capture *company performance and health*. Fusing them should outperform either stream alone.

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle, warnings, os
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

df = pd.read_csv('../data/df_processed.csv')
print(f'Loaded: {df.shape}')

/Users/mk2004/Downloads/CATIVE_fixed/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Device: cpu
Loaded: (1000, 30)


## 1. Tabular Feature Preparation (reuse Phase 1 preprocessing)

In [2]:
sector_map = {
    'AI / Dev Tools': 'AI', 'Semiconductor / Edge AI': 'Semiconductor',
    'DevOps / Observability': 'DevTools', 'Defence / Anti-Drone': 'DeepTech',
    'Defence Tech': 'DeepTech', 'DeepTech / Aviation': 'DeepTech',
    'Industrial Robotics': 'Robotics', 'Underwater Robotics': 'Robotics',
    'CleanTech / Battery Materials': 'Climate Tech', 'GreenTech / Hydrogen': 'Climate Tech',
    'AgriTech / EV': 'AgriTech', 'HR Tech / Global Mobility': 'HR Tech',
    'D2C / FMCG': 'D2C', 'Gaming Hardware / D2C': 'D2C', 'F&B / QSR': 'D2C',
    'Food Delivery': 'D2C', 'Home Services': 'SaaS', 'Quick Commerce Infra': 'Logistics',
}
df['sector_clean'] = df['sector'].replace(sector_map)
city_freq = df['hq_city'].value_counts()
rare_cities = city_freq[city_freq < 5].index.tolist()
df['city_clean'] = df['hq_city'].apply(lambda x: 'Other' if x in rare_cities else x)

funding_order = {'Bootstrapped': 0, 'Seed': 1, 'Pre-Series A': 2, 'Series A': 3, 'Series B': 4}
df['funding_ordinal'] = df['funding_stage'].map(funding_order)

# Engineered features
df['hiring_pressure']      = df['active_job_openings'] / (df['employee_count'] + 1)
df['culture_wlb_score']    = (df['culture_rating'] + df['wlb_rating']) / 2
df['growth_signal']        = df['employee_growth_pct'] * (1 + df['ai_roles_focus'])
df['funding_per_employee'] = np.log1p(df['total_funding_usd']) / (df['employee_count'] + 1)
df['startup_age']          = 2025 - df['founded_year']
df['review_density']       = df['glassdoor_reviews_count'] / (df['employee_count'] + 1)

NUMERIC_FEATS = [
    'employee_count', 'total_funding_usd', 'employee_growth_pct',
    'active_job_openings', 'wlb_rating', 'culture_rating',
    'growth_opportunity_rating', 'salary_rating', 'overall_employee_rating',
    'glassdoor_reviews_count', 'layoff_flag', 'ai_roles_focus', 'tier2_expansion',
    'funding_ordinal', 'hiring_pressure', 'culture_wlb_score', 'growth_signal',
    'funding_per_employee', 'startup_age', 'review_density',
]

remote_dummies = pd.get_dummies(df['remote_friendly'], prefix='remote', drop_first=True)
sector_clean_dummies = pd.get_dummies(df['sector_clean'], prefix='sec', drop_first=True)
city_clean_dummies   = pd.get_dummies(df['city_clean'],   prefix='city', drop_first=True)

X_tab = pd.concat([
    df[NUMERIC_FEATS],
    remote_dummies, sector_clean_dummies, city_clean_dummies
], axis=1).values.astype(np.float32)

le = LabelEncoder()
y_enc = le.fit_transform(df['hire_quality_label'])

# Scale numeric features
scaler = StandardScaler()
X_tab[:, :len(NUMERIC_FEATS)] = scaler.fit_transform(X_tab[:, :len(NUMERIC_FEATS)])

# --- Integrate GMM Posteriors (Phase 3 Optimization) ---
X_full  = pd.read_csv('../data/X_full.csv')
pca_gmm = pickle.load(open('../outputs/models/pca_gmm.pkl', 'rb'))
gmm     = pickle.load(open('../outputs/models/gmm_model.pkl', 'rb'))
# Use .values if column names are strictly required to be absent or handled by PCA
gmm_probs = gmm.predict_proba(pca_gmm.transform(X_full.values)).astype(np.float32)

X_tab = np.hstack([X_tab, gmm_probs])

print(f'Tabular feature matrix: {X_tab.shape}')
print(f'Feature dim for DL tabular stream: {X_tab.shape[1]}')

Tabular feature matrix: (1000, 53)
Feature dim for DL tabular stream: 53


## 2. DistilBERT Tokenisation

In [3]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

MAX_LEN = 32  # role strings are short; avg ~5 tokens
roles_text = df['top_hiring_roles'].str.replace(';', ',').tolist()

encodings = tokenizer(
    roles_text,
    max_length=MAX_LEN,
    padding='max_length',
    truncation=True,
    return_tensors='np'
)

input_ids      = encodings['input_ids'].astype(np.int64)
attention_mask = encodings['attention_mask'].astype(np.int64)

print(f'input_ids shape: {input_ids.shape}')
print(f'Sample tokenised (first row): {tokenizer.decode(input_ids[0])}')

input_ids shape: (1000, 32)
Sample tokenised (first row): [CLS] software engineer, qa executive, ops manager [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


## 3. Stratified Train/Val/Test Split (70/15/15)

In [4]:
idx = np.arange(len(y_enc))
idx_trainval, idx_test = train_test_split(idx, test_size=0.15, stratify=y_enc, random_state=SEED)
idx_train, idx_val     = train_test_split(idx_trainval, test_size=0.176, stratify=y_enc[idx_trainval], random_state=SEED)

print(f'Train: {len(idx_train)}, Val: {len(idx_val)}, Test: {len(idx_test)}')
print(f'Train class dist: {np.bincount(y_enc[idx_train])}')
print(f'Val class dist:   {np.bincount(y_enc[idx_val])}')
print(f'Test class dist:  {np.bincount(y_enc[idx_test])}')

# Save for Phase 2 notebooks
np.save('../data/dl_idx_train.npy', idx_train)
np.save('../data/dl_idx_val.npy',   idx_val)
np.save('../data/dl_idx_test.npy',  idx_test)
np.save('../data/dl_X_tab.npy',     X_tab)
np.save('../data/dl_input_ids.npy', input_ids)
np.save('../data/dl_attn_mask.npy', attention_mask)
np.save('../data/dl_y.npy',         y_enc)
np.save('../data/dl_tab_dim.npy',   np.array([X_tab.shape[1]]))
pickle.dump(scaler, open('../outputs/models/dl_scaler.pkl','wb'))
pickle.dump(le,     open('../outputs/models/dl_label_encoder.pkl','wb'))
print('All DL data artefacts saved (including updated TAB_DIM).')

Train: 700, Val: 150, Test: 150
Train class dist: [233 234 233]
Val class dist:   [50 50 50]
Test class dist:  [50 50 50]
All DL data artefacts saved (including updated TAB_DIM).


## 4. PyTorch Dataset Class

In [5]:
class StartupDataset(Dataset):
    def __init__(self, input_ids, attention_mask, X_tab, y):
        self.input_ids      = torch.tensor(input_ids,      dtype=torch.long)
        self.attention_mask = torch.tensor(attention_mask, dtype=torch.long)
        self.X_tab          = torch.tensor(X_tab,          dtype=torch.float32)
        self.y              = torch.tensor(y,               dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'tabular':        self.X_tab[idx],
            'label':          self.y[idx]
        }

# Create splits
train_ds = StartupDataset(input_ids[idx_train], attention_mask[idx_train], X_tab[idx_train], y_enc[idx_train])
val_ds   = StartupDataset(input_ids[idx_val],   attention_mask[idx_val],   X_tab[idx_val],   y_enc[idx_val])
test_ds  = StartupDataset(input_ids[idx_test],  attention_mask[idx_test],  X_tab[idx_test],  y_enc[idx_test])

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')
# Quick sanity check
batch = next(iter(train_loader))
print(f'Batch input_ids: {batch["input_ids"].shape}')
print(f'Batch tabular: {batch["tabular"].shape}')
print(f'Batch labels: {batch["label"].shape}')

TAB_DIM = X_tab.shape[1]
print(f'\nTabular dim for DL model: {TAB_DIM}')
np.save('../data/dl_tab_dim.npy', np.array([TAB_DIM]))

Train batches: 22, Val batches: 5, Test batches: 5
Batch input_ids: torch.Size([32, 32])
Batch tabular: torch.Size([32, 53])
Batch labels: torch.Size([32])

Tabular dim for DL model: 53
